In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install datasets

In [3]:
from datasets import load_dataset

In [4]:
TRAIN_PATH ="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

In [5]:
data = load_dataset("csv",data_files=TRAIN_PATH)

Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
train = data["train"]

In [8]:
train

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})

In [9]:
def combine(row):
    new_row = row.copy()
    
    row["combined_text"] = row["prompt"] + " "+ row["A"]
    return row 
    

In [11]:
train[80]

{'id': 81,
 'prompt': 'Determine the correct option: What is the purpose of obtaining surgical resection specimens? based on the given context.',
 'A': 'To remove an entire diseased area or organ for definitive surgical treatment of a disease, with pathological analysis of the specimen used to confirm the diagnosis.',
 'B': 'To perform visual and microscopic tests on tissue samples using automated analysers and cultures.',
 'C': 'To work in close collaboration with medical technologists and hospital administrations.',
 'D': 'To administer a variety of tests of the biophysical properties of tissue samples.',
 'E': 'To obtain bodily fluids such as blood and urine for laboratory analysis of disease diagnosis.',
 'answer': 'A'}

In [13]:
combined_data = train.map(combine)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [14]:
print(len(combined_data[51]["combined_text"]))

614


Q1-AND= 614

In [18]:
train["prompt"]

Column(["Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.", 'What is accelerator-based light-ion fusion?', 'Determine the correct option: What is the term used in astrophysics to describe light-matter interactions resulting in energy shifts in the radiation field? among the listed options.', "Select the most accurate option: What is Martin Heidegger's view on the relationship between time and human existence? carefully.", "Identify the correct statement: What is the concept of simultaneity in Einstein's book, Relativity? carefully.", ...])

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
print(tokenizer.vocab_size)

30522


In [17]:
print(tokenizer.convert_tokens_to_ids("[SEP]"))

102


In [24]:
encoded_prompt = tokenizer(
    list(train["prompt"]),
    padding="max_length",
    max_length=128,
    truncation=True,
    return_tensors="pt"
)

In [26]:
print(encoded_prompt["input_ids"].shape)

torch.Size([2000, 128])


In [23]:
type(list(train["prompt"]))

list

In [30]:
from transformers import AutoConfig , AutoModel

config = AutoConfig.from_pretrained("bert-base-uncased")
print(config.hidden_size)          
print(config.num_attention_heads)   

head_dim = config.hidden_size // config.num_attention_heads
print(head_dim)           

768
12
64


In [33]:
encode_first =tokenizer(
    train[0]["prompt"],
    return_tensors="pt"
)

In [31]:
model = AutoModel.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:

out = model(**encode_first)

In [36]:
out.last_hidden_state.shape

torch.Size([1, 31, 768])

In [40]:
out.last_hidden_state[0,0][:5].sum().item()

-1.2000964879989624

In [41]:
encoded2 = tokenizer(
    ["Light-ion fusion is a technique",],
    return_tensors = "pt"
)

In [53]:
out2 = model(**encoded2,output_attentions=True)

In [54]:
tokens = tokenizer.convert_ids_to_tokens(encoded2["input_ids"][0])
print(tokens)

['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '[SEP]']


In [55]:
out2

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.3780, -0.5899, -0.1764,  ..., -0.0345,  0.0343,  0.5419],
         [-0.6363,  0.5274, -0.1328,  ...,  0.0331,  0.5034,  0.2527],
         [-0.1802,  0.6585,  0.2120,  ..., -0.3016,  0.1367,  0.2934],
         ...,
         [-0.4564, -0.3809, -0.0915,  ..., -0.3090, -0.6339,  0.9510],
         [-0.3712,  0.0414, -0.3501,  ..., -0.4763, -0.0970, -0.0720],
         [ 0.7096,  0.0619, -0.2556,  ...,  0.1992, -0.8412, -0.2903]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[-7.4680e-01,  1.2755e-03,  2.5116e-01,  2.0580e-01,  5.8114e-02,
         -8.0981e-03, -1.3103e-01,  1.8086e-02,  4.4624e-01, -9.9906e-01,
          6.1331e-02, -1.3081e-01,  9.6038e-01, -5.6525e-01,  5.4044e-01,
         -3.0451e-01,  2.7663e-01, -3.0778e-01,  1.6218e-02,  4.6798e-01,
          6.3601e-02,  9.7567e-01,  5.7413e-01,  1.6778e-01, -1.5469e-02,
         -2.6515e-01, -2.6906e-01,  7.5506e-01,  8.4906e-01,  5.676

In [58]:
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [59]:
print(model.config.output_attentions)

True


In [60]:
out = model(**encoded2)

In [62]:
last_layer = out.attentions[-1]      

head0 = last_layer[0, 0]                

fusion_idx = tokens.index("fusion")

attention = head0[0, fusion_idx].item()

print(round(attention, 4))

0.1041


In [63]:
from sentence_transformers import SentenceTransformer, util


model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prompt = train[0]["prompt"]
option_b = train[0]["B"]


prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

similarity = util.cos_sim(prompt_embedding, option_b_embedding)

print(similarity)
print(round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

tensor([[0.7658]])
0.7658


In [64]:
from sklearn.metrics.pairwise import cosine_similarity

In [66]:
train_copy = pd.read_csv(TRAIN_PATH)

In [67]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

documents = (
    train_copy["prompt"].fillna("") + " " +
    train_copy["A"].fillna("") + " " +
    train_copy["B"].fillna("") + " " +
    train_copy["C"].fillna("") + " " +
     train_copy["D"].fillna("") + " " + train_copy["E"]
).tolist()

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(documents)

In [69]:
cols = ["A","B","C","D","E"]
sm =0.0
all_preds = []
for i in range(train_copy.shape[0]):
    prompt_vec = vectorizer.transform([train_copy['prompt'].iloc[i]])
    sims = np.zeros(5)
    for j in range(5):
        option_vec = vectorizer.transform([train_copy[cols[j]].iloc[i]])
        sim = cosine_similarity(prompt_vec,option_vec)[0][0]
        sims[j] = sim
    sorted_idx = np.argsort(sims)[::-1]
    corr = train_copy["answer"].iloc[i]
    all_preds.append(sorted_idx[:3])
    if  corr == cols[sorted_idx[0]]:
        sm += 1.0
    elif corr == cols[sorted_idx[1]]:
        sm += 0.5 
    elif corr == cols[sorted_idx[2]]:
        sm += 1/3
    
    
print(sm/train_copy.shape[0])

0.2552499999999985


In [72]:
all_preds_allmini = []
sm =0.0 
for i in range(train_copy.shape[0]):
    prompt_embedding = model.encode(train_copy["prompt"].iloc[i], convert_to_tensor=True)
    sims = np.zeros(5)
    for j in range(5):
        opt = model.encode(train_copy[cols[j]].iloc[i],convert_to_tensor= True)
        similarity = util.cos_sim(prompt_embedding, opt)
        sims[j] = similarity
    sorted_idx = np.argsort(sims)[::-1]
    corr = train_copy["answer"].iloc[i]
    all_preds_allmini.append(sorted_idx[:3])
    if  corr == cols[sorted_idx[0]]:
        sm += 1.0
    elif corr == cols[sorted_idx[1]]:
        sm += 0.5 
    elif corr == cols[sorted_idx[2]]:
        sm += 1/3

print(sm/train_copy.shape[0])

0.42308333333333487


In [80]:
labels = {v:i for i ,v in enumerate(cols)}

In [87]:
train_copy

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A
...,...,...,...,...,...,...,...,...
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,E
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,D
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,B


In [81]:
labels

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

In [85]:

an = 0  
for i in range(train_copy.shape[0]):
    corr = train_copy["answer"].iloc[i]
    l = labels[corr]
   
    if l not in all_preds[i]:
        if l in all_preds_allmini[i]:
            an += 1 


In [86]:
print(an)

613


In [ ]:
OPTION_COLS = ["A","B","C","D","E"]
